# Storage Solution Test

This notebook tests `storage.solution.py` independently with a temporary SQLite database.

It covers:
- creating the table
- adding expenses
- querying and filtering expenses
- updating and deleting expenses
- verifying missing-record errors


In [ ]:
import importlib.util
import pathlib

base = pathlib.Path('.')
spec = importlib.util.spec_from_file_location('storage_solution', base / 'storage.solution.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
ExpenseStorage = module.ExpenseStorage

path = pathlib.Path('tmp_solution_test.db')
if path.exists():
    path.unlink()

store = ExpenseStorage(str(path))
first = store.add_expense('groceries', 45.0, 'weekly shopping', '2026-06-30')
second = store.add_expense('coffee', 4.5)
all_rows = store.get_expenses()
filtered = store.get_expenses(category='groceries')
updated = store.update_expense(first['id'], amount=50.0)
removed = store.delete_expense(second['id'])

results = {
    'first': first,
    'second': second,
    'all_count': len(all_rows),
    'filtered_count': len(filtered),
    'updated': updated,
    'removed': removed,
}

try:
    store.update_expense(9999, amount=1.0)
except Exception as exc:
    results['missing_update'] = f'{type(exc).__name__}: {exc}'

try:
    store.delete_expense(9999)
except Exception as exc:
    results['missing_delete'] = f'{type(exc).__name__}: {exc}'

results